# DIV 12 LFP coherence over distance with Figure 2 burst anchors

This notebook is a diagnostic-first scaffold for reproducing the paper's coherence-over-distance analysis on the DIV 12 `stimRemovalNull` LFP recording, while deliberately using the event-anchor method already used for Figure 2.

The paper anchored burst spectra on maximal summed nLFP activity. Here the burst intervals and anchors are spike-derived: high-activity epochs are detected from smoothed population IFR, participation bursts are detected inside those epochs, and each burst is anchored at the maximum high-resolution population IFR. The coherence window is the detected burst interval itself, not the paper's `-0.33*Tburst` to `+0.67*Tburst` window.

In [ ]:
%matplotlib inline

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
import os
import pickle
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ephax.metrics.burst import (
    assign_max_population_ifr_burst_anchors,
    build_highres_traces,
    build_participation_activity_state,
    build_population_ifr,
    detect_high_activity_epochs,
    detect_participation_burst_epochs,
)
from ephax.metrics.lfp import (
    bandpass_downsample,
    dataset_for_well,
    extract_phasors,
    inspect_file,
    load_chunk,
    load_data_store_spikes,
)
from ephax.preprocessing.dataset import Recording
from ephax.plotting import PAPER_COLORS, apply_paper_style
from ephax.plotting.style import LINE_WIDTHS

apply_paper_style()
plt.rcParams["figure.dpi"] = 170
np.random.seed(0)

## Configuration

Defaults are intentionally capped. Set `FORCE_RECOMPUTE = True` after changing analysis parameters, and increase `MAX_EVENTS_PER_WELL` only after the anchor diagnostics look sensible.

In [ ]:
DIV = 12
WELLS = [0, 1, 2, 3, 4, 5]
EXAMPLE_WELL = 0
TOP_START = 0
TOP_STOP = 100
MIN_AMP = 0.0

IFR_GRID_HZ = 50.0
SMOOTH_SIGMA_SEC = 0.15
HIGH_ACTIVITY_MAD_SCALE = 3.0
HIGH_ACTIVITY_MIN_DURATION_MS = 30.0
HIGH_ACTIVITY_MAX_GAP_BINS = 0
HIGHRES_BIN_MS = 1.0
HIGHRES_SMOOTH_SIGMA_MS = 3.0
NETWORK_BIN_MS = 10.0
NETWORK_MIN_PARTICIPATION_FRACTION = 0.05
NETWORK_MIN_DURATION_MS = 20.0

FS_DS = 500.0
FILTER_CONTEXT_PAD_S = 0.35
FREQ_LOW_HZ = 4.0
FREQ_HIGH_HZ = 100.0
N_FREQUENCIES = int(os.environ.get("LFP_COH_N_FREQUENCIES", "32"))
REL_BANDWIDTH = 0.22
MAX_EVENTS_PER_WELL = int(os.environ.get("LFP_COH_MAX_EVENTS_PER_WELL", "20"))
SHIFT_REPS = int(os.environ.get("LFP_COH_SHIFT_REPS", "5"))
BACKGROUND_SHIFT_MAX_S = 5.0
BACKGROUND_TRIES = 200
DISTANCE_BIN_UM = 100.0
DISTANCE_MAX_UM = 3500.0
BOOTSTRAP_REPS = int(os.environ.get("LFP_COH_BOOTSTRAP_REPS", "1000"))
RANDOM_SEED = 0
FORCE_RECOMPUTE = False
SAVE_TABLES = True

RAW_LFP_FILE = Path("/Users/danielrebbin/Documents/Academia/UvA/Internship/Wes_Files/Data/LFP/stimRemovalNull") / f"DIV {DIV}" / "data.raw.h5"
SMOKE_MODE = os.environ.get("LFP_COH_SMOKE", "0") == "1"
OUTPUT_DIR = repo_root / "outputs" / "lfp_coherence_distance_div12_fig2_anchor_top100"
if SMOKE_MODE:
    OUTPUT_DIR = OUTPUT_DIR / "smoke"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
CACHE_PATH = OUTPUT_DIR / "coherence_distance_cache.pkl"
for _path in (OUTPUT_DIR, TABLE_DIR, FIGURE_DIR):
    _path.mkdir(parents=True, exist_ok=True)

if SMOKE_MODE:
    WELLS = [EXAMPLE_WELL]
    MAX_EVENTS_PER_WELL = min(MAX_EVENTS_PER_WELL, 2)
    SHIFT_REPS = min(SHIFT_REPS, 1)
    N_FREQUENCIES = min(N_FREQUENCIES, 8)
    BOOTSTRAP_REPS = min(BOOTSTRAP_REPS, 50)

CONFIG = {
    "div": DIV,
    "wells": WELLS,
    "top_start": TOP_START,
    "top_stop": TOP_STOP,
    "frequency_range_hz": (FREQ_LOW_HZ, FREQ_HIGH_HZ),
    "n_frequencies": N_FREQUENCIES,
    "rel_bandwidth": REL_BANDWIDTH,
    "max_events_per_well": MAX_EVENTS_PER_WELL,
    "shift_reps": SHIFT_REPS,
    "anchor_method": "max_highres_population_ifr",
    "window_method": "detected_burst_interval",
}
CONFIG

## Event anchors

This section recreates the Figure 2 event-selection path on the raw LFP data-store file. The final `anchor_time_s` is the maximum high-resolution population IFR within each participation burst.

In [ ]:
def _recording_for_well(well: int) -> tuple[Recording, dict[str, object]]:
    dataset = dataset_for_well(well)
    info = inspect_file(RAW_LFP_FILE, dataset)
    spikes, layout, sf = load_data_store_spikes(RAW_LFP_FILE, dataset, min_amp=MIN_AMP)
    raw_stop_s = float(info["raw_shape"][1]) / float(info["fs_raw"])
    spike_stop_s = float(np.nanmax(spikes["time"])) if len(spikes.get("time", [])) else raw_stop_s
    stop_s = min(raw_stop_s, spike_stop_s)
    recording = Recording(spikes=spikes, layout=layout, start_time=0.0, end_time=stop_s, sf=float(sf))
    return recording, info


def detect_well_burst_anchors(well: int) -> dict[str, object]:
    recording, info = _recording_for_well(well)
    refs = recording.refs_top(start=TOP_START, stop=TOP_STOP)
    population = build_population_ifr(
        recording,
        refs,
        grid_hz=IFR_GRID_HZ,
        smooth_sigma_sec=SMOOTH_SIGMA_SEC,
    )
    high_epochs, high_info = detect_high_activity_epochs(
        population.time_grid,
        population.mean_ifr_smooth,
        mad_scale=HIGH_ACTIVITY_MAD_SCALE,
        min_duration_ms=HIGH_ACTIVITY_MIN_DURATION_MS,
        max_gap_bins=HIGH_ACTIVITY_MAX_GAP_BINS,
    )
    highres = build_highres_traces(
        recording,
        refs,
        bin_ms=HIGHRES_BIN_MS,
        smooth_sigma_ms=HIGHRES_SMOOTH_SIGMA_MS,
    )
    participation = build_participation_activity_state(highres, aggregation_ms=NETWORK_BIN_MS)
    bursts = detect_participation_burst_epochs(
        participation,
        high_epochs,
        min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
        min_duration_ms=NETWORK_MIN_DURATION_MS,
    )
    bursts = assign_max_population_ifr_burst_anchors(highres, bursts)
    bursts = bursts.sort_values("anchor_time_s").reset_index(drop=True)
    bursts.insert(0, "well", int(well))
    bursts.insert(1, "dataset", dataset_for_well(well))
    bursts["raw_file"] = str(RAW_LFP_FILE)
    bursts["n_selected_refs"] = int(len(refs))
    return {
        "well": int(well),
        "recording": recording,
        "info": info,
        "refs": np.asarray(refs, dtype=int),
        "population": population,
        "high_epochs": high_epochs,
        "high_info": high_info,
        "highres": highres,
        "participation": participation,
        "bursts": bursts,
    }


def build_anchor_contexts(wells=WELLS):
    contexts = {}
    for well in wells:
        t0 = perf_counter()
        contexts[int(well)] = detect_well_burst_anchors(int(well))
        bursts = contexts[int(well)]["bursts"]
        print(
            f"well {well}: refs={len(contexts[int(well)]['refs'])}, "
            f"high={len(contexts[int(well)]['high_epochs'])}, bursts={len(bursts)}, "
            f"elapsed={perf_counter() - t0:.1f}s"
        )
    return contexts


anchor_contexts = build_anchor_contexts(WELLS)
anchor_table = pd.concat([ctx["bursts"] for ctx in anchor_contexts.values()], ignore_index=True)
anchor_table["anchor_inside_burst"] = (
    (anchor_table["anchor_time_s"] >= anchor_table["start_time_s"])
    & (anchor_table["anchor_time_s"] <= anchor_table["end_time_s"])
)
print(anchor_table[["well", "event_idx", "start_time_s", "end_time_s", "anchor_time_s", "duration_ms", "peak_active_electrodes", "anchor_inside_burst"]].head().to_string(index=False))
assert bool(anchor_table["anchor_inside_burst"].all()), "At least one anchor falls outside its burst interval."
anchor_table

In [ ]:
def plot_well_anchor_diagnostics(well: int):
    ctx = anchor_contexts[int(well)]
    population = ctx["population"]
    bursts = ctx["bursts"]
    high_epochs = ctx["high_epochs"]
    fig, ax = plt.subplots(figsize=(8.0, 2.8), constrained_layout=True)
    ax.plot(population.time_grid, population.mean_ifr_smooth, color="black", lw=0.8, label="smoothed population IFR")
    for row in high_epochs.itertuples(index=False):
        ax.axvspan(float(row.start_time_s), float(row.end_time_s), color=PAPER_COLORS["high_activity"], alpha=0.18, linewidth=0)
    for row in bursts.itertuples(index=False):
        ax.axvspan(float(row.start_time_s), float(row.end_time_s), color=PAPER_COLORS["burst"], alpha=0.22, linewidth=0)
        ax.axvline(float(row.anchor_time_s), color=PAPER_COLORS["highlight"], lw=0.8, alpha=0.8)
    ax.set(title=f"DIV {DIV} well {well}: Figure 2 burst anchors", xlabel="Time (s)", ylabel="IFR (Hz)")
    ax.legend(loc="upper right", frameon=False)
    return fig, ax

fig, ax = plot_well_anchor_diagnostics(EXAMPLE_WELL)

## Coherence helpers

The first pass uses log-spaced, band-limited analytic phase coherence. This is wavelet-style in intent: short event windows are analyzed across a log frequency grid. No fixed 58-62 Hz exclusion is applied.

In [ ]:
def make_log_frequency_bands(low_hz=FREQ_LOW_HZ, high_hz=FREQ_HIGH_HZ, n_frequencies=N_FREQUENCIES, rel_bandwidth=REL_BANDWIDTH, *, fs_ds=FS_DS):
    centers = np.geomspace(float(low_hz), float(high_hz), int(n_frequencies))
    nyquist = 0.5 * float(fs_ds)
    rows = []
    for center in centers:
        half_width = 0.5 * float(rel_bandwidth) * float(center)
        lo = max(0.25, float(center) - half_width)
        hi = min(0.98 * nyquist, float(center) + half_width)
        if hi > lo:
            rows.append({"frequency_hz": float(center), "low_hz": lo, "high_hz": hi, "band": frequency_band_name(center)})
    return pd.DataFrame(rows)


def frequency_band_name(freq_hz: float) -> str:
    freq_hz = float(freq_hz)
    if 4.0 <= freq_hz < 15.0:
        return "theta"
    if 15.0 <= freq_hz < 30.0:
        return "beta"
    if 30.0 <= freq_hz <= 100.0:
        return "gamma"
    return "other"


def selected_channel_indices(mapping: dict[str, np.ndarray], selected_electrodes: np.ndarray) -> np.ndarray:
    electrodes = np.asarray(mapping["electrode"], dtype=int)
    selected = np.asarray(selected_electrodes, dtype=int)
    idx = np.flatnonzero(np.isin(electrodes, selected))
    if idx.size == 0:
        raise ValueError("No selected spike electrodes mapped to LFP channels.")
    return idx


def _interval_overlaps(start_s: float, stop_s: float, intervals: pd.DataFrame) -> bool:
    if intervals is None or intervals.empty:
        return False
    start = float(start_s)
    stop = float(stop_s)
    return bool(((intervals["end_time_s"].astype(float) > start) & (intervals["start_time_s"].astype(float) < stop)).any())


def shifted_background_windows(event_row, all_bursts: pd.DataFrame, recording_stop_s: float, *, reps=SHIFT_REPS, rng=None):
    rng = np.random.default_rng(RANDOM_SEED) if rng is None else rng
    event_start = float(event_row["start_time_s"])
    event_stop = float(event_row["end_time_s"])
    duration = event_stop - event_start
    if duration <= 0:
        return []
    windows = []
    for _ in range(int(reps)):
        accepted = None
        for _try in range(BACKGROUND_TRIES):
            shift = rng.uniform(-BACKGROUND_SHIFT_MAX_S, BACKGROUND_SHIFT_MAX_S)
            if abs(shift) < duration:
                continue
            start = event_start + shift
            stop = start + duration
            if start < 0 or stop > recording_stop_s:
                continue
            if _interval_overlaps(start, stop, all_bursts):
                continue
            accepted = (float(start), float(stop))
            break
        if accepted is not None:
            windows.append(accepted)
    return windows


def distance_bin_table(coords_um: np.ndarray, ref_idx: int, target_indices: np.ndarray):
    coords_um = np.asarray(coords_um, dtype=float)
    ref_xy = coords_um[int(ref_idx)]
    target_xy = coords_um[np.asarray(target_indices, dtype=int)]
    distances = np.linalg.norm(target_xy - ref_xy[None, :], axis=1)
    edges = np.arange(0.0, float(DISTANCE_MAX_UM) + float(DISTANCE_BIN_UM), float(DISTANCE_BIN_UM))
    bin_idx = np.digitize(distances, edges) - 1
    centers = 0.5 * (edges[:-1] + edges[1:])
    return distances, bin_idx, centers

In [ ]:
def load_selected_lfp_window(well: int, selected_refs: np.ndarray, start_s: float, stop_s: float):
    dataset = dataset_for_well(well)
    info = inspect_file(RAW_LFP_FILE, dataset)
    fs_raw = float(info["fs_raw"])
    recording_stop_s = float(info["raw_shape"][1]) / fs_raw
    read_start_s = max(0.0, float(start_s) - FILTER_CONTEXT_PAD_S)
    read_stop_s = min(recording_stop_s, float(stop_s) + FILTER_CONTEXT_PAD_S)
    start_frame = int(round(read_start_s * fs_raw))
    n_frames = max(1, int(round((read_stop_s - read_start_s) * fs_raw)))
    raw, coords_mm, fs_raw, mapping = load_chunk(RAW_LFP_FILE, dataset, start_frame, n_frames)
    sel_idx = selected_channel_indices(mapping, selected_refs)
    raw = raw[:, sel_idx]
    coords_mm = coords_mm[sel_idx]
    selected_mapping = {key: np.asarray(value)[sel_idx] for key, value in mapping.items()}
    return {
        "raw": raw,
        "coords_mm": coords_mm,
        "coords_um": coords_mm * 1000.0,
        "fs_raw": float(fs_raw),
        "mapping": selected_mapping,
        "read_start_s": read_start_s,
        "read_stop_s": read_stop_s,
        "target_start_s": float(start_s),
        "target_stop_s": float(stop_s),
    }


def choose_reference_channel(window_data, *, gamma_low=30.0, gamma_high=80.0):
    raw = window_data["raw"]
    if raw.shape[1] == 1:
        return 0
    band = bandpass_downsample(raw, window_data["fs_raw"], gamma_low, gamma_high, FS_DS, max_channels_per_block=128)
    t_ds = window_data["read_start_s"] + np.arange(band.shape[0]) / FS_DS
    target = (t_ds >= window_data["target_start_s"]) & (t_ds <= window_data["target_stop_s"])
    if not np.any(target):
        target[:] = True
    _, amp = extract_phasors(band)
    score = np.nanmean(amp[target], axis=0)
    if not np.any(np.isfinite(score)):
        return 0
    return int(np.nanargmax(score))


def compute_window_coherence(well: int, event_idx: int, selected_refs: np.ndarray, start_s: float, stop_s: float, *, condition: str, shift_rep=np.nan, ref_idx: int | None = None):
    window = load_selected_lfp_window(well, selected_refs, start_s, stop_s)
    if window["raw"].shape[1] < 2:
        raise ValueError("Need at least two selected LFP channels for coherence.")
    if ref_idx is None:
        ref_idx = choose_reference_channel(window)
    ref_electrode = int(window["mapping"]["electrode"][ref_idx])
    target_indices = np.asarray([idx for idx in range(window["raw"].shape[1]) if idx != int(ref_idx)], dtype=int)
    distances_um, distance_bin_idx, distance_centers = distance_bin_table(window["coords_um"], int(ref_idx), target_indices)
    freq_bands = make_log_frequency_bands()
    freq_rows = []
    site_rows = []
    for freq_row in freq_bands.itertuples(index=False):
        band = bandpass_downsample(
            window["raw"],
            window["fs_raw"],
            float(freq_row.low_hz),
            float(freq_row.high_hz),
            FS_DS,
            max_channels_per_block=128,
        )
        t_ds = window["read_start_s"] + np.arange(band.shape[0]) / FS_DS
        target = (t_ds >= window["target_start_s"]) & (t_ds <= window["target_stop_s"])
        if np.count_nonzero(target) < 3:
            continue
        u, amp = extract_phasors(band)
        pair_phase = u[target][:, target_indices] * np.conj(u[target][:, [int(ref_idx)]])
        site_coh = np.abs(np.nanmean(pair_phase, axis=0))
        ref_amp = float(np.nanmean(amp[target, int(ref_idx)]))
        finite = np.isfinite(site_coh)
        if np.any(finite):
            freq_rows.append({
                "well": int(well),
                "event_idx": int(event_idx),
                "condition": condition,
                "shift_rep": shift_rep,
                "frequency_hz": float(freq_row.frequency_hz),
                "band": str(freq_row.band),
                "coherence_mean": float(np.nanmean(site_coh[finite])),
                "coherence_median": float(np.nanmedian(site_coh[finite])),
                "n_sites": int(np.count_nonzero(finite)),
                "ref_electrode": ref_electrode,
                "ref_amplitude": ref_amp,
                "target_start_s": float(start_s),
                "target_stop_s": float(stop_s),
            })
        for target_i, distance_um, bin_i, coh in zip(target_indices, distances_um, distance_bin_idx, site_coh):
            if not np.isfinite(coh) or bin_i < 0 or bin_i >= len(distance_centers):
                continue
            site_rows.append({
                "well": int(well),
                "event_idx": int(event_idx),
                "condition": condition,
                "shift_rep": shift_rep,
                "frequency_hz": float(freq_row.frequency_hz),
                "band": str(freq_row.band),
                "electrode": int(window["mapping"]["electrode"][target_i]),
                "ref_electrode": ref_electrode,
                "distance_um": float(distance_um),
                "distance_bin_idx": int(bin_i),
                "distance_bin_center_um": float(distance_centers[int(bin_i)]),
                "coherence": float(coh),
                "target_start_s": float(start_s),
                "target_stop_s": float(stop_s),
            })
    return pd.DataFrame(freq_rows), pd.DataFrame(site_rows), int(ref_idx), ref_electrode

## Event-level coherence

The event selector ranks bursts by peak active electrode count and population rate so the diagnostic set starts with robust anchors.

In [ ]:
def select_events_for_well(ctx: dict[str, object], max_events=MAX_EVENTS_PER_WELL) -> pd.DataFrame:
    bursts = ctx["bursts"].copy()
    if bursts.empty:
        return bursts
    sort_cols = [col for col in ["peak_active_electrodes", "anchor_population_rate_hz", "duration_ms"] if col in bursts.columns]
    selected = bursts.sort_values(sort_cols, ascending=False).head(int(max_events)).sort_values("anchor_time_s")
    return selected.reset_index(drop=True)


def compute_well_coherence(ctx: dict[str, object], *, max_events=MAX_EVENTS_PER_WELL, shift_reps=SHIFT_REPS, rng=None):
    rng = np.random.default_rng(RANDOM_SEED + int(ctx["well"])) if rng is None else rng
    well = int(ctx["well"])
    selected_events = select_events_for_well(ctx, max_events=max_events)
    selected_refs = np.asarray(ctx["refs"], dtype=int)
    recording_stop_s = float(ctx["info"]["raw_shape"][1]) / float(ctx["info"]["fs_raw"])
    freq_frames = []
    site_frames = []
    selected_event_rows = []
    for row in selected_events.itertuples(index=False):
        event_idx = int(row.event_idx)
        event_start = float(row.start_time_s)
        event_stop = float(row.end_time_s)
        if event_stop <= event_start:
            continue
        print(f"well {well} event {event_idx}: burst {event_start:.3f}-{event_stop:.3f}s")
        freq_df, site_df, ref_idx, ref_electrode = compute_window_coherence(
            well,
            event_idx,
            selected_refs,
            event_start,
            event_stop,
            condition="burst",
            shift_rep=np.nan,
            ref_idx=None,
        )
        freq_frames.append(freq_df)
        site_frames.append(site_df)
        selected_event_rows.append({
            "well": well,
            "event_idx": event_idx,
            "anchor_time_s": float(row.anchor_time_s),
            "start_time_s": event_start,
            "end_time_s": event_stop,
            "duration_s": event_stop - event_start,
            "ref_electrode": int(ref_electrode),
            "n_selected_refs": int(len(selected_refs)),
            "peak_active_electrodes": int(getattr(row, "peak_active_electrodes", -1)),
            "anchor_population_rate_hz": float(getattr(row, "anchor_population_rate_hz", np.nan)),
        })
        bg_windows = shifted_background_windows(row._asdict(), ctx["bursts"], recording_stop_s, reps=shift_reps, rng=rng)
        for shift_rep, (bg_start, bg_stop) in enumerate(bg_windows):
            bg_freq_df, bg_site_df, _, _ = compute_window_coherence(
                well,
                event_idx,
                selected_refs,
                bg_start,
                bg_stop,
                condition="shifted",
                shift_rep=int(shift_rep),
                ref_idx=ref_idx,
            )
            freq_frames.append(bg_freq_df)
            site_frames.append(bg_site_df)
    return {
        "selected_events": pd.DataFrame(selected_event_rows),
        "frequency_coherence": pd.concat(freq_frames, ignore_index=True) if freq_frames else pd.DataFrame(),
        "site_distance_coherence": pd.concat(site_frames, ignore_index=True) if site_frames else pd.DataFrame(),
    }


def compute_all_coherence(contexts: dict[int, dict[str, object]]):
    all_selected = []
    all_freq = []
    all_site = []
    for well, ctx in contexts.items():
        t0 = perf_counter()
        product = compute_well_coherence(ctx)
        all_selected.append(product["selected_events"])
        all_freq.append(product["frequency_coherence"])
        all_site.append(product["site_distance_coherence"])
        print(f"well {well}: coherence elapsed={perf_counter() - t0:.1f}s")
    return {
        "config": CONFIG,
        "anchor_table": anchor_table,
        "selected_events": pd.concat(all_selected, ignore_index=True) if all_selected else pd.DataFrame(),
        "frequency_coherence": pd.concat(all_freq, ignore_index=True) if all_freq else pd.DataFrame(),
        "site_distance_coherence": pd.concat(all_site, ignore_index=True) if all_site else pd.DataFrame(),
    }


coherence_product = None
if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    with CACHE_PATH.open("rb") as fh:
        cached_product = pickle.load(fh)
    if cached_product.get("config") == CONFIG:
        coherence_product = cached_product
        print(f"Loaded cached coherence product: {CACHE_PATH}")
    else:
        print("Ignoring coherence cache with non-matching config.")
if coherence_product is None:
    coherence_product = compute_all_coherence(anchor_contexts)
    with CACHE_PATH.open("wb") as fh:
        pickle.dump(coherence_product, fh)
    print(f"Saved coherence product: {CACHE_PATH}")

selected_events = coherence_product["selected_events"]
frequency_coherence = coherence_product["frequency_coherence"]
site_distance_coherence = coherence_product["site_distance_coherence"]
print({"selected_events": len(selected_events), "frequency_rows": len(frequency_coherence), "site_rows": len(site_distance_coherence)})
selected_events.head()

## Summaries and diagnostics

In [ ]:
def summarize_frequency_coherence(freq_df: pd.DataFrame) -> pd.DataFrame:
    if freq_df.empty:
        return pd.DataFrame()
    return (
        freq_df.groupby(["condition", "frequency_hz", "band"], as_index=False)
        .agg(
            coherence_mean=("coherence_mean", "mean"),
            coherence_median=("coherence_median", "median"),
            n_events=("event_idx", "nunique"),
        )
        .sort_values(["condition", "frequency_hz"])
    )


def summarize_distance_coherence(site_df: pd.DataFrame) -> pd.DataFrame:
    if site_df.empty:
        return pd.DataFrame()
    grouped = (
        site_df.groupby(["condition", "band", "well", "distance_bin_center_um"], as_index=False)
        .agg(
            coherence_mean=("coherence", "mean"),
            coherence_median=("coherence", "median"),
            n_events=("event_idx", "nunique"),
            n_sites_total=("coherence", "size"),
        )
        .sort_values(["condition", "band", "well", "distance_bin_center_um"])
    )
    return grouped


def relative_distance_summary(site_df: pd.DataFrame) -> pd.DataFrame:
    if site_df.empty:
        return pd.DataFrame()
    event_summary = (
        site_df.groupby(["well", "event_idx", "condition", "band", "distance_bin_center_um"], as_index=False)
        .agg(coherence=("coherence", "mean"))
    )
    burst = event_summary[event_summary["condition"] == "burst"].drop(columns="condition")
    shifted = (
        event_summary[event_summary["condition"] == "shifted"]
        .groupby(["well", "event_idx", "band", "distance_bin_center_um"], as_index=False)
        .agg(shifted=("coherence", "mean"))
    )
    merged = burst.merge(shifted, on=["well", "event_idx", "band", "distance_bin_center_um"], how="inner")
    merged = merged.rename(columns={"coherence": "burst"})
    merged["coherence_relative"] = merged["burst"] / merged["shifted"].replace(0, np.nan)
    rows = []
    rng = np.random.default_rng(RANDOM_SEED)
    for keys, group in merged.groupby(["band", "distance_bin_center_um"]):
        band, distance = keys
        values = group["coherence_relative"].to_numpy(float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        boot = []
        for _ in range(int(BOOTSTRAP_REPS)):
            sample = rng.choice(values, size=values.size, replace=True)
            boot.append(float(np.nanmedian(sample)))
        ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan)
        rows.append({
            "band": band,
            "distance_bin_center_um": float(distance),
            "coherence_relative_mean": float(np.nanmean(values)),
            "coherence_relative_median": float(np.nanmedian(values)),
            "coherence_relative_ci_low": float(ci_low),
            "coherence_relative_ci_high": float(ci_high),
            "n_wells": int(group["well"].nunique()),
            "n_events": int(group[["well", "event_idx"]].drop_duplicates().shape[0]),
        })
    return pd.DataFrame(rows).sort_values(["band", "distance_bin_center_um"])


frequency_summary = summarize_frequency_coherence(frequency_coherence)
distance_summary = summarize_distance_coherence(site_distance_coherence)
relative_summary = relative_distance_summary(site_distance_coherence)

if SAVE_TABLES:
    anchor_table.to_csv(TABLE_DIR / "event_anchors.csv", index=False)
    selected_events.to_csv(TABLE_DIR / "selected_events.csv", index=False)
    frequency_coherence.to_csv(TABLE_DIR / "frequency_coherence.csv", index=False)
    site_distance_coherence.to_csv(TABLE_DIR / "site_distance_coherence.csv", index=False)
    frequency_summary.to_csv(TABLE_DIR / "frequency_summary.csv", index=False)
    distance_summary.to_csv(TABLE_DIR / "condition_distance_summary.csv", index=False)
    relative_summary.to_csv(TABLE_DIR / "relative_distance_summary.csv", index=False)
    pd.DataFrame([CONFIG]).to_csv(TABLE_DIR / "method_config.csv", index=False)
    print(f"Saved tables to {TABLE_DIR}")

frequency_summary.head(), distance_summary.head(), relative_summary.head()

In [ ]:
def plot_frequency_summary(summary: pd.DataFrame):
    if summary.empty:
        print("No frequency summary to plot.")
        return None
    fig, ax = plt.subplots(figsize=(6.4, 3.2), constrained_layout=True)
    for condition, group in summary.groupby("condition"):
        group = group.sort_values("frequency_hz")
        color = PAPER_COLORS["burst"] if condition == "burst" else PAPER_COLORS["low_activity"]
        ax.plot(group["frequency_hz"], group["coherence_mean"], marker="o", ms=2.8, lw=1.0, label=condition, color=color)
    ax.set_xscale("log")
    ax.set(xlabel="Frequency (Hz)", ylabel="Mean reference-site coherence", title="Burst vs shifted spectral coherence")
    ax.legend(frameon=False)
    return fig


def plot_relative_distance(summary: pd.DataFrame):
    if summary.empty:
        print("No relative distance summary to plot.")
        return None
    fig, ax = plt.subplots(figsize=(6.4, 3.2), constrained_layout=True)
    colors = {"theta": "tab:blue", "beta": "tab:green", "gamma": PAPER_COLORS["burst"], "other": "0.5"}
    for band, group in summary.groupby("band"):
        if band == "other":
            continue
        group = group.sort_values("distance_bin_center_um")
        x = group["distance_bin_center_um"].to_numpy(float)
        y = group["coherence_relative_median"].to_numpy(float)
        lo = group["coherence_relative_ci_low"].to_numpy(float)
        hi = group["coherence_relative_ci_high"].to_numpy(float)
        color = colors.get(str(band), None)
        ax.plot(x, y, marker="o", ms=3.0, lw=1.0, label=str(band), color=color)
        ax.fill_between(x, lo, hi, color=color, alpha=0.16, linewidth=0)
    ax.axhline(1.0, color="0.2", lw=0.8, ls="--")
    ax.set(xlabel="Distance from reference electrode (um)", ylabel="Burst / shifted coherence", title="Relative coherence over distance")
    ax.legend(frameon=False)
    return fig

fig_freq = plot_frequency_summary(frequency_summary)
fig_dist = plot_relative_distance(relative_summary)
if fig_freq is not None:
    fig_freq.savefig(FIGURE_DIR / "frequency_coherence_burst_vs_shifted.png", dpi=300)
if fig_dist is not None:
    fig_dist.savefig(FIGURE_DIR / "relative_coherence_over_distance.png", dpi=300)

## Example event heatmap

This diagnostic checks whether the burst period looks meaningfully different from nearby non-burst background before interpreting aggregate distance curves.

In [ ]:
def compute_example_event_heatmap(well=EXAMPLE_WELL):
    events = selected_events[selected_events["well"] == int(well)].copy()
    if events.empty:
        print(f"No selected events for well {well}.")
        return None
    event = events.sort_values(["peak_active_electrodes", "anchor_population_rate_hz"], ascending=False).iloc[0]
    ctx = anchor_contexts[int(well)]
    window = load_selected_lfp_window(int(well), ctx["refs"], float(event.start_time_s), float(event.end_time_s))
    ref_idx = choose_reference_channel(window)
    bands = make_log_frequency_bands()
    rows = []
    t_target = None
    for row in bands.itertuples(index=False):
        band = bandpass_downsample(window["raw"], window["fs_raw"], float(row.low_hz), float(row.high_hz), FS_DS, max_channels_per_block=128)
        t_ds = window["read_start_s"] + np.arange(band.shape[0]) / FS_DS
        mask = (t_ds >= window["target_start_s"]) & (t_ds <= window["target_stop_s"])
        u, _ = extract_phasors(band)
        coherence_t = np.abs(np.nanmean(u[:, np.arange(u.shape[1]) != ref_idx] * np.conj(u[:, [ref_idx]]), axis=1))
        rows.append(coherence_t[mask])
        if t_target is None:
            t_target = t_ds[mask]
    if not rows or t_target is None:
        return None
    return {
        "well": int(well),
        "event_idx": int(event.event_idx),
        "anchor_time_s": float(event.anchor_time_s),
        "start_time_s": float(event.start_time_s),
        "end_time_s": float(event.end_time_s),
        "ref_electrode": int(window["mapping"]["electrode"][ref_idx]),
        "time_s": t_target,
        "frequencies_hz": bands["frequency_hz"].to_numpy(float),
        "coherence": np.vstack(rows),
    }


def draw_example_heatmap(data):
    if data is None:
        return None
    fig, ax = plt.subplots(figsize=(6.6, 2.8), constrained_layout=True)
    mesh = ax.pcolormesh(data["time_s"], data["frequencies_hz"], data["coherence"], shading="auto", cmap="viridis", vmin=0, vmax=1)
    ax.axvline(data["anchor_time_s"], color=PAPER_COLORS["highlight"], lw=LINE_WIDTHS["base"], ls="--")
    ax.set_yscale("log")
    ax.set(xlabel="Time (s)", ylabel="Frequency (Hz)", title=f"Well {data['well']} event {data['event_idx']}: event-window coherence")
    cbar = fig.colorbar(mesh, ax=ax, pad=0.015)
    cbar.set_label("Phase coherence")
    return fig

example_heatmap = compute_example_event_heatmap(EXAMPLE_WELL)
fig_heatmap = draw_example_heatmap(example_heatmap)
if fig_heatmap is not None:
    fig_heatmap.savefig(FIGURE_DIR / "example_event_frequency_time_coherence.png", dpi=300)
example_heatmap

## Acceptance checks

These checks are intentionally simple and visible. If burst and shifted coherence look indistinguishable, inspect the anchor table and example heatmap before increasing event counts.

In [ ]:
checks = {
    "anchors_nonempty": len(anchor_table) > 0,
    "anchors_inside_bursts": bool(anchor_table["anchor_inside_burst"].all()) if len(anchor_table) else False,
    "selected_events_nonempty": len(selected_events) > 0,
    "frequency_rows_nonempty": len(frequency_coherence) > 0,
    "site_distance_rows_nonempty": len(site_distance_coherence) > 0,
}
checks